In [0]:
# !pip install ibm-aigov-facts-client
# !pip install python-dotenv
!pip install tensorflow-cpu

In [0]:
%restart_python

In [0]:
import os
from dotenv import load_dotenv
import requests, json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense
import mlflow
import mlflow.tensorflow


from ibm_aigov_facts_client import AIGovFactsClient, CloudPakforDataConfig, DetachedPromptTemplate, PromptTemplate, DeploymentDetails, ModelDetails

In [0]:
model_name= "amazon-stock-price-0918"
print(f"model/{model_name}.keras")

In [0]:
model= keras.saving.load_model(f"model/{model_name}.keras")
model

## Deploy model on Databricks

In [0]:
# Enable MLflow autologging
mlflow.tensorflow.autolog()

mlflow.set_registry_uri("databricks")

with mlflow.start_run() as run:
    mlflow.tensorflow.log_model(model=model, artifact_path= "model")


In [0]:
import mlflow

mlflow.set_registry_uri("databricks")

model_uri = f"runs:/{run.info.run_id}/model"
model_details = mlflow.register_model(model_uri=model_uri,name="AmazonStockLSTM-0919")

In [0]:
# Create a serving endpoint from the registered model
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name="AmazonStockLSTM-0919",
    version=model_details.version,
    stage="Production",
)


## Promote to Deployment

In [0]:
endpoint_name=f"{model_name}-endpoint"
endpoint_url = "https://adb-3528367123873887.7.azuredatabricks.net/serving-endpoints/amazonstock/invocations"
deployment_name = f"{model_name} Deployment"

deployment=DeploymentDetails(identifier=endpoint_name,name=deployment_name,deployment_type="online",scoring_endpoint=endpoint_url,description="Model deployed in Azure databricks to predict Amazon stock prices")

model_details = ModelDetails(model_type="TensorFlow",input_type="Structured", algorithm="LSTM",label_column="High", label_type="float",prediction_type="Regression")

In [0]:
inventory_id="e24ff319-4ee8-4716-a346-bb6e23383167" # TD Model Inventory

training_data_reference = {
    "id": "amzn_stock_updated_v_2",            
    "type": "fs",                             
    "location": {
        "path": "dbfs:/user/hive/warehouse/amzn_stock_updated_v_2",
        "connection": {
            "type": "dbfs"
        }
    },
    "data_type": "csv",
    "description": "Training data stored in DBFS for LSTM model"
}

In [0]:
load_dotenv(".env")
CPD_URL = os.getenv("CPD_URL")
CPD_USERNAME = os.getenv("CPD_USERNAME")
CPD_PASSWORD = os.getenv("CPD_PASSWORD")

facts_client = AIGovFactsClient(
    cloud_pak_for_data_configs=creds,
    experiment_name=model_name,
    external_model=True,
    enable_autolog=False,
    set_as_current_experiment=True
)

In [0]:
external_model=facts_client.external_model_facts.save_external_model_asset(model_identifier= model_name,
                                                                           name=model_name,
                                                                           model_details=model_details,
                                                                           training_data_reference=training_data_reference,
                                                                           deployment_details=deployment,
                                                                           description="Model developed in Azure databricks to predict Amazon Stock prices",
                                                                           catalog_id=inventory_id)

## Promote to Validate

In [0]:
model_id= "b0e02a14-f1a6-4cab-bb5d-45622184c247"
model= facts_client.assets.get_model(model_id=model_id, container_id= inventory_id)

In [0]:
model.set_environment_type(from_container="test",to_container="validate")

## Promote to Operate

In [0]:
model_id= "b0e02a14-f1a6-4cab-bb5d-45622184c247"
model= facts_client.assets.get_model(model_id=model_id, container_id= inventory_id)

In [0]:
model.set_environment_type(from_container="validate",to_container="operate")